# 1장 1강: 문제 유형·데이터 분리·평가지표 리마인드

## 실습 목표

- Ames Housing의 `SalePrice`를 예측하는 문제가 어떤 머신러닝 문제 유형인지 구분합니다.
- 입력 데이터 `X`와 정답 `y`를 구분합니다.
- 데이터를 `train / valid / test`로 분리하고 각 데이터의 역할을 확인합니다.
- 회귀 평가지표 `MSE`, `RMSE`, `MAE`, `R²`의 의미를 확인합니다.
- 교안에서 소개한 **평균 예측 baseline**을 기준으로 기본 성능을 해석합니다.

> **실습 범위 안내**  
> 이번 실습은 1장 1강 교안의 리마인드 범위만 사용합니다.  
> 별도의 회귀 알고리즘, 교차검증, 하이퍼파라미터 튜닝, 새로운 평가지표는 사용하지 않습니다.

## 실습 환경 / 데이터

- Python
- pandas, numpy, scikit-learn
- 데이터: `Ames_Housing.csv`
- 예측 대상: `SalePrice` (주택 판매 가격)
- 이번 실습에서 확인할 입력 컬럼: `GrLivArea`, `OverallQual`, `YearBuilt`


## 실습 준비

먼저 데이터를 불러온 뒤 이번 실습에서 사용할 컬럼을 확인합니다.

- `GrLivArea`: 지상 생활 면적
- `OverallQual`: 주택의 전반적인 품질
- `YearBuilt`: 건축 연도
- `SalePrice`: 주택 판매 가격

이번 강의의 핵심은 **문제 유형, 데이터 분리, 평가지표**이므로 복잡한 전처리는 진행하지 않습니다.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../Ames_Housing.csv")

use_cols = ["GrLivArea", "OverallQual", "YearBuilt", "SalePrice"]
data = df[use_cols].copy()

print("데이터 크기:", data.shape)
display(data.head())


데이터 크기: (1460, 4)


,GrLivArea,OverallQual,YearBuilt,SalePrice
0,1710,7,2003,208500
1,1262,6,1976,181500
2,1786,7,2001,223500
3,1717,7,1915,140000
4,2198,8,2000,250000


In [2]:
print("[자료형]")
display(data.dtypes.to_frame("dtype"))

print("[결측치 수]")
display(data.isna().sum().to_frame("missing_count"))

print("[기초 통계량]")
display(data.describe())


[자료형]


,dtype
GrLivArea,int64
OverallQual,int64
YearBuilt,int64
SalePrice,int64


[결측치 수]


,missing_count
GrLivArea,0
OverallQual,0
YearBuilt,0
SalePrice,0


[기초 통계량]


,GrLivArea,OverallQual,YearBuilt,SalePrice
count,1460.000000,1460.000000,1460.000000,1460.000000
mean,1515.463699,6.099315,1971.267808,180921.195890
std,525.480383,1.382997,30.202904,79442.502883
min,334.000000,1.000000,1872.000000,34900.000000
25%,1129.500000,5.000000,1954.000000,129975.000000
50%,1464.000000,6.000000,1973.000000,163000.000000
75%,1776.750000,7.000000,2000.000000,214000.000000
max,5642.000000,10.000000,2010.000000,755000.000000


---

# 필수 1. 문제 유형과 train / valid / test 분리

## 배경

Ames Housing 데이터에서 주택의 특성을 이용해 `SalePrice`를 예측하려고 합니다.

교안에서 머신러닝 문제는 회귀·분류·군집화·차원축소로 구분했고, 지도학습에서는 입력 `X`와 정답 `y`를 사용합니다. 또한 모델의 일반화 성능을 확인하기 위해 데이터를 `train / valid / test`로 나눕니다.

## 문제

### 요구사항

1. `SalePrice` 예측이 회귀·분류·군집화·차원축소 중 어떤 문제인지 작성하세요.
2. 아래 기준으로 `X`와 `y`를 만드세요.
   - X: `GrLivArea`, `OverallQual`, `YearBuilt`
   - y: `SalePrice`
3. 교안의 예시와 같이 데이터를 약 **6 : 2 : 2** 비율의 `train / valid / test`로 나누세요.
   - 첫 번째 분리: `test_size=0.2`, `random_state=42`
   - 두 번째 분리: 남은 데이터에서 `test_size=0.25`, `random_state=42`
4. `X_train`, `X_valid`, `X_test`의 행 개수를 출력하세요.
5. 아래 질문에 답하세요.

### Q&A

- **Q1.** `SalePrice` 예측은 왜 해당 문제 유형에 속하나요?
- **Q2.** train 데이터는 어떤 목적으로 사용하나요?
- **Q3.** valid 데이터는 어떤 목적으로 사용하나요?
- **Q4.** test 데이터는 어떤 목적으로 사용하나요?

### 확인 포인트

- 정답 `y`가 존재하는 지도학습 문제인지 확인했는가?
- `SalePrice`가 연속적인 숫자 값이라는 점을 근거로 문제 유형을 판단했는가?
- train / valid / test의 역할을 구분했는가?


In [ ]:
from sklearn.model_selection import train_test_split

# TODO 1. X와 y를 만드세요.
X = data[["GrLivArea", "OverallQual", "YearBuilt"]] # 필요한 변수들만 선별
y = data["SalePrice"]

print("X 크기:", X.shape, "| y 크기:", y.shape)
print("결측치 확인:\n", data.isna().sum().to_string())

# TODO 2. train+valid / test로 먼저 분리하세요.

# 1차 분리: 전체 → (train+valid) 80% / test 20%
X_temp, X_test, y_temp, y_test = train_test_split( # temp(train+valid)와 test로 분리
    X, y, test_size=0.2, random_state=42
)

# TODO 3. 남은 데이터를 train / valid로 분리하세요.

X_train, X_valid, y_train, y_valid = train_test_split( #temp를 train과 valid로 분리
    X_temp, y_temp, test_size=0.25, random_state=42
)

# TODO 4. 각 데이터의 행 개수를 출력하세요.

total = len(X)
print(f"X_train: {len(X_train)}행 ({len(X_train)/total*100:.1f}%)") # 학습지 (실제 가격의 패턴을 학습)
print(f"X_valid: {len(X_valid)}행 ({len(X_valid)/total*100:.1f}%)") # 모의고사 (train에서 정한 평균으로 예측한 뒤 성능 확인)
print(f"X_test : {len(X_test)}행 ({len(X_test)/total*100:.1f}%)") # 실제 시험 (최종 성능 확인)
print(f"합계   : {len(X_train)+len(X_valid)+len(X_test)}행")


X 크기: (1460, 3) | y 크기: (1460,)
결측치 확인:
 GrLivArea      0
OverallQual    0
YearBuilt      0
SalePrice      0
X_train: 876행 (60.0%)
X_valid: 292행 (20.0%)
X_test : 292행 (20.0%)
합계   : 1460행


### 필수 1 답변 작성

- **Q1. 문제 유형과 이유:**  
-> `회귀, SalePrice는 주택 판매 가격이라는 연속적인 숫자 값을 예측하는 대상`  
- **Q2. train의 역할:**  
-> `모델이 데이터의 패턴을 학습하는데 사용`  
- **Q3. valid의 역할:**  
-> `여러 모델이나 설정(하이퍼 파라미터 튜닝)이 좋은지 검증하고 선택하는데 사용`    
- **Q4. test의 역할:**  
-> `처음 보는 데이터에서도 학습한 모델의 최종 성능이 잘 작동하는지 테스트하기 위함`


---

# 필수 2. 평균 예측 baseline과 회귀 평가지표

## 배경

교안에서는 회귀 문제를 `MSE`, `RMSE`, `MAE`, `R²`로 평가할 수 있다고 배웠습니다. 또한 R²를 설명하면서 **아무 특성도 사용하지 않고 y의 평균값만 예측하는 기준 모델(baseline)**을 소개했습니다.

이번 문제에서는 새로운 모델 알고리즘을 사용하지 않고, **train의 SalePrice 평균을 모든 valid 데이터의 예측값으로 사용하는 baseline**을 만들어 네 가지 회귀 평가지표를 확인합니다.

## 문제

### 요구사항

1. `y_train`의 평균을 `baseline_value`에 저장하세요.
2. `baseline_value`를 valid 데이터 개수만큼 반복하여 `y_valid_pred`를 만드세요.
3. `MSE`, `RMSE`, `MAE`, `R²`를 계산하세요.
4. 네 가지 결과를 출력하세요.
5. 아래 질문에 답하세요.

### Q&A

- **Q1.** MSE는 무엇을 의미하나요?
- **Q2.** RMSE는 무엇을 의미하며 MSE와 어떤 차이가 있나요?
- **Q3.** MAE는 무엇을 의미하나요?
- **Q4.** R²가 0에 매우 가깝다면 이 baseline을 어떻게 해석할 수 있나요?
- **Q5.** RMSE가 MAE보다 크게 나오는 이유를 교안의 설명을 이용해 작성하세요.

### 확인 포인트

- baseline 값은 반드시 `y_train`의 평균으로 만들었는가?
- MSE, RMSE, MAE, R²만 사용했는가?
- RMSE가 큰 오차에 더 큰 영향을 받는다는 점을 설명했는가?


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TODO 1. y_train 평균으로 baseline 값을 만드세요.
baseline_value = y_train.mean()

# TODO 2. valid 데이터 전체에 같은 baseline 값을 예측하도록 만드세요.
y_valid_pred = np.full(len(y_valid), baseline_value)

# TODO 3. MSE, RMSE, MAE, R²를 계산하세요.

# mean_squared_error: 실제값과 예측값의 차이를 제곱하여 평균 mse를 계산
mse = mean_squared_error(y_valid, y_valid_pred) # y_valid_pred(정답지)를 보고, y_valid로 모의고사 봐서 잘 맞는지 검증해봐야함

# RMSE 계산
rmse = np.sqrt(mse)

# mean_absolute_error: 차이의 절대값을 평균한 MAE를 계산
mae = mean_absolute_error(y_valid, y_valid_pred)

# R2: 평가 데이터의 평균을 예측하는 기준과 비교하여 R2값을 계산
r2 = r2_score(y_valid, y_valid_pred)

# TODO 4. 결과를 출력하세요.
print(f"baseline 값 (y_train 평균): {baseline_value:,.2f}")
print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"MAE : {mae:,.2f}")
print(f"R²  : {r2:.4f}")

baseline 값 (y_train 평균): 181,233.97
MSE : 5,989,110,885.72
RMSE: 77,389.35
MAE : 57,793.60
R²  : -0.0001


# 해설  
- BaseLine: 181,233 / 달러 / 모든 검증 주택에 동일하게 제시한 예측 가격  
- MSE: 5,989,110 / 달러² / 제곱한 오차의 평균(낮을수록 좋음) / 약 주택 한 채당 59억달러 정도의 오차를 보임  
- RMSE: 77,389 / 달러 / 큰 오차를 더 강하게 반영한 오차 규모(낮을수록 좋음) / 약 주택 한 대당 77,389달러 정도의 오차를 낼 수 있음
- MAE: 57,793 / 달러 / 절대 오차의 평균(낮을수록 좋음) / 전체적으로 57,793달러 정도 차이가 날 수 있음임 (RMSE와 다름)
- R²: -0.00 / 단위 없음 / 평가 데이터 평균을 예측하는 기준에 비해 얼마나 개선되었는가?(높을수록 좋음), 설명력 / -0.0001만큼 설명하고 있음. 지금 예측을 전혀 못하고 있다는 뜻  
-> `이 BaseLine은 면적이나 품질, 건축연도를 사용하지 않아 주택별 차이를 충분히 반영하지 못함.`  
   `완성된 가격 예측 방법의 우수성을 보여주는 결과가 아니라 이후 방법을 평가할 기준 성능을 의미`  
   `학습 데이터의 평균 가격인 181,233달러를 모든 검증 주택 예측에 사용했고, 검증 MAE는 약 57,793달러, RMSE는 약 77,389달러, R²은 -0.00으로 데이터 평균을 예측하는 기준과 거의 같은 수준`

### 필수 2 답변 작성

- **Q1. MSE:**  
->`실제값과 예측값의 오차를 제곱한 뒤 평균한 값, 큰 오차가 있으면 숫자가 커짐`  

- **Q2. RMSE:**  
-> `MSE에 제곱을 취한 값. 원래 단위로 해석하여 오차를 직관적으로 볼 수 있음`  

- **Q3. MAE:**  
-> `실제값과 예측값 차이의 절댓값을 평균한 값. 평균적으로 얼마나 벗어났는지를 보여줌`  

- **Q4. R² 해석:**  
-> `특성을 사용하지 않고 평균값만 예측하는 기준 모델 수준과 다르지 않음`  
- **Q4-1. R²가 음수나 0에 가깝다는 의미 해석:**  
-> R²는 높을수록 좋은데, 음수나 0에 가깝다는 이야기는 모델링은 진행가능하나  
-> 모델 자체가 해당 데이터에 대한 설명력이 0에 가깝거나 음수로 표현된다.  
-> `즉, 데이터를 충분히 설명할 수 있는 모델은 아니다.`  

- **Q5. RMSE와 MAE 차이 해석:**  
-> `RMSE는 오차를 제곱하여 빗나간(크게 틀린) 경우에 더 민감하게 반음`  
-> `큰 오차가 존재하면 MAE보다 더 값이 크게 나올 수 있음`  


---

# 과제 1. 테스트 데이터에서 baseline 최종 확인

## 배경

교안에서는 test 데이터를 **최종 성능을 마지막에 한 번 확인하는 데이터**라고 설명했습니다.

필수 2에서 사용한 평균 예측 baseline을 그대로 유지한 상태에서 test 데이터의 성능을 확인해보세요. 새로운 모델이나 새로운 평가 방법은 사용하지 않습니다.

## 문제

### 요구사항

1. 필수 2에서 만든 `baseline_value`를 그대로 사용하세요.
2. test 데이터 개수만큼 동일한 값을 예측하여 `y_test_pred`를 만드세요.
3. test 데이터에 대해 `MSE`, `RMSE`, `MAE`, `R²`를 계산하세요.
4. 결과를 출력하세요.
5. 아래 질문에 답하세요.

### Q&A

- **Q1.** 이번 과제에서 test 데이터를 사용하는 목적은 무엇인가요?
- **Q2.** test의 R²가 0에 가깝거나 음수라면 평균 예측 baseline의 성능을 어떻게 해석할 수 있나요?
- **Q3.** RMSE와 MAE 중 큰 오차에 더 큰 벌점을 주는 지표는 무엇이며, 왜 그런가요?

### 확인 포인트

- test 데이터에서 새로운 값을 학습하거나 기준을 다시 정하지 않았는가?
- `y_train`에서 구한 baseline 값을 그대로 사용했는가?
- 교안의 회귀 평가지표 범위만 사용했는가?


In [ ]:
# TODO 1. test 데이터의 baseline 예측값을 만드세요.

# 주의: baseline_value는 필수 2에서 y_train으로 구한 값을 그대로 사용
#       y_test.mean()으로 다시 계산하면 test 데이터를 학습에 쓰는 것이 됨

y_test_pred = np.full(len(y_test), baseline_value)

# TODO 2. test 데이터의 MSE, RMSE, MAE, R²를 계산하세요.
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

# TODO 3. 결과를 출력하세요.
print(f"사용한 baseline 값 (y_train 평균): {baseline_value:,.2f}")
print(f"[test] MSE : {mse_test:,.2f}")
print(f"[test] RMSE: {rmse_test:,.2f}")
print(f"[test] MAE : {mae_test:,.2f}")
print(f"[test] R²  : {r2_test:.4f}")


사용한 baseline 값 (y_train 평균): 181,233.97
[test] MSE : 7,676,058,192.02
[test] RMSE: 87,613.12
[test] MAE : 62,516.21
[test] R²  : -0.0007


### 과제 1 답변 작성

- **Q1. test 데이터의 목적:**  
→ `완전히 새로운 데이터에서 어느 정도의 성능을 내는지 최종 성능 평가를 하기 위해 test 데이터를 사용`  
`이때 test로 기준값을 다시 정하거나 결과를 보고 모델을 수정하면 이미 학습이 되었기 때문에 처음보는 데이터에서의 성능이라는 의미를 잃게됨`  

- **Q2. R² 해석:**  
- `R²: -0.0007로 음수로 나옴`  
→ `모델이 -0.0007만큼 설명하고 있음 충분히 설명력이 있는 모델이라고 보기 힘들다.`  
- **Q3. 큰 오차에 더 큰 벌점을 주는 지표와 이유:**  
→ `RMSE`  
`RMS는 오차를 제곱하여 평균한 뒤 제곱근을 적용하는 지표로 큰 오차에 더 큰 벌점을 준다.`  
`반면 MAE는 절댓값의 평균이기 때문에 RMSE가 큰 오차에 더 크게 영향을 받는다.`  


---

# 실습 마무리

아래 세 가지를 한 문장씩 정리해보세요.

1. **어떤 분석 문제가 있었는가?**  
분류, 회귀, 군집화 중 어느 것을 적용해야하는지 확인이 필요했고  
`-> SalesPrice를 예측해야하는 회귀 문제였다.`  
2. **어떤 데이터 분리와 평가지표를 적용했는가?**  
`데이터 분리: Train / Valid / Test`  
`평가지표: MAE, MSE, RMSE, R²`  

3. **평균 예측 baseline의 결과를 무엇을 근거로 해석했는가?**  
-> `Train의 평균값을 사용하는 baseline의 valid, test 평가 결과와 각 회귀 성능 평가지표의 의미를 근거로 기준 성능을 확인`
